# 02 — Fine-tuning Cross-Encoder: Eksp. 1 & Eksp. 3 (+ efisiensi)

Notebook ini melatih **pointwise cross-encoder** `[CLS] query [SEP] passage [SEP]` → skor relevansi, lalu me-rerank top-100 BM25 pada **Dev resmi (960 kueri)**.

| Eksperimen | Run | Data latih | Seed |
|---|---|---|---|
| **Eksp. 1** (RQ1–2) | mBERT, mDeBERTa | 100% (3.664 kueri) | 42, 100, 2026 |
| **Eksp. 3** (RQ4) | mBERT, mDeBERTa | 25% & 50% | 42 |

Hyperparameter **sama untuk kedua model, tanpa tuning** (slide 11): AdamW, BCE, lr 2e-5, warmup 0,1, weight decay 0,01, batch 16, 3 epoch, max length 256, BF16. Checkpoint dipilih dari epoch dengan nDCG@10 tertinggi di 10% validasi.

**Yang disimpan per run** (ke Drive, tanpa checkpoint model):
- `scores/{run}.dev_top100.parquet` — skor cross-encoder untuk seluruh 100 kandidat BM25 per kueri Dev (cukup untuk Eksp. 2, karena top-20/50 adalah subset top-100 dan skor pointwise tidak bergantung kandidat lain),
- `results/{run}.json` — metrik Dev, riwayat validasi, waktu training,
- `results/{run}.per_query.csv` — metrik per kueri (untuk uji signifikansi).

Setelah run **seed 42 dengan 100% data**, efisiensi inferensi (latency, throughput, memori) untuk k ∈ {20, 50, 100} langsung diukur → `results/efficiency_{model}.json`.

> ⏱️ Setiap run ±20–40 menit di L4. Run yang hasilnya sudah ada di Drive otomatis dilewati, jadi jika Colab terputus cukup jalankan ulang semua sel.

**Prasyarat:** notebook 01 sudah dijalankan. **Runtime:** Runtime ▸ Change runtime type ▸ **L4 GPU**.

## 1. Setup

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!pip -q install pytrec-eval-terrier sentencepiece protobuf

name, memory.total [MiB], driver_version
NVIDIA L4, 23034 MiB, 580.82.07


In [2]:
# Mount Google Drive: semua data, skor, dan hasil disimpan di sini agar aman
# bila sesi Colab terputus.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

BASE_DIR    = '/content/drive/MyDrive/MIRACL_Reranking'   # ubah bila perlu
DATA_DIR    = f'{BASE_DIR}/data'      # topics, qrels, BM25 run, passages, training pairs
SCORES_DIR  = f'{BASE_DIR}/scores'    # skor cross-encoder per run (Dev top-100)
RESULTS_DIR = f'{BASE_DIR}/results'   # metrik (JSON), tabel (CSV), log
FIG_DIR     = f'{BASE_DIR}/figures'   # grafik (PNG)
for d in [DATA_DIR, SCORES_DIR, RESULTS_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)
print('BASE_DIR =', BASE_DIR)

BASE_DIR = /content/drive/MyDrive/MIRACL_Reranking


In [2]:
import gc, json, math, os, random, time
os.environ['TOKENIZERS_PARALLELISM'] = 'false'   # tokenisasi di DataLoader worker
import numpy as np, pandas as pd, torch
import transformers
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16 = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu'
print(f'torch {torch.__version__} | transformers {transformers.__version__} | '
      f'device {GPU_NAME} | BF16 {USE_BF16}')
if 'L4' not in GPU_NAME:
    print('⚠️  GPU bukan L4 — angka efisiensi tidak akan sebanding dengan proposal.')

torch 2.11.0+cu130 | transformers 5.20.2 | device NVIDIA L4 | BF16 True


### Environment Version

In [3]:
import sys, platform, importlib.metadata as md
print('Python      :', sys.version.split()[0], '|', platform.platform())
for pkg in ['pyserini', 'pytrec-eval-terrier', 'torch', 'transformers', 'sentencepiece',
            'numpy', 'pandas', 'scipy', 'scikit-learn', 'pyarrow', 'matplotlib']:
    try: print(f'{pkg:20s}:', md.version(pkg))
    except md.PackageNotFoundError: print(f'{pkg:20s}: (tidak terinstal)')
!java -version 2>&1 | head -1
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!nvcc --version | tail -1
!lscpu | grep "Model name"; free -h | grep Mem

Python      : 3.13.16 | Linux-6.6.159+-x86_64-with-glibc2.39
pyserini            : (tidak terinstal)
pytrec-eval-terrier : 0.5.11
torch               : 2.11.0+cu130
transformers        : 5.20.2
sentencepiece       : 0.2.2
numpy               : 2.1.3
pandas              : 2.2.3
scipy               : 1.16.3
scikit-learn        : 1.6.1
pyarrow             : 23.0.1
matplotlib          : 3.10.0
[0.013s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
NVIDIA L4, 23034 MiB, 580.82.07
Build cuda_13.0.r13.0/compiler.36424714_0
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Mem:            52Gi       3.8Gi        44Gi       3.7Mi       5.2Gi        49Gi


## 2. Konfigurasi eksperimen

In [5]:
EXPERIMENT = 'exp3'          # 'exp1' (3 seed, 100% data) atau 'exp3' (25%/50%, seed 42)
RUN_ONLY   = None            # opsional, mis. [('mdeberta', 42, 1.0)] untuk menjalankan satu run saja
FORCE_RERUN = False          # True = timpa hasil yang sudah ada

MODELS = {
    'mbert':    'bert-base-multilingual-cased',
    'mdeberta': 'microsoft/mdeberta-v3-base',
}
HP = dict(
    lr=2e-5, weight_decay=0.01, warmup_ratio=0.1,
    batch_size=16, epochs=3, max_len=256,
    max_grad_norm=1.0,         # default Transformers Trainer
    eval_batch_size=128,       # untuk scoring validasi & Dev (tidak diukur waktunya)
)
EXP1_SEEDS = [42, 100, 2026]
EXP3_FRACS = [0.25, 0.50]
EFF_K_LIST = [20, 50, 100]    # Eksp. 2: latency/throughput diukur setelah run seed 42 (100%)
EFF_INFER_BS = 32             # batch size inferensi TETAP saat mengukur latency
EFF_WARMUP_QUERIES = 20       # kueri awal yang tidak dihitung (warmup)
LOG_EVERY = 200

def planned_runs():
    if RUN_ONLY:
        return list(RUN_ONLY)
    if EXPERIMENT == 'exp1':
        return [(m, s, 1.0) for m in MODELS for s in EXP1_SEEDS]
    if EXPERIMENT == 'exp3':
        return [(m, 42, f) for m in MODELS for f in EXP3_FRACS]
    raise ValueError(EXPERIMENT)

def run_name(model_key, seed, frac):
    return f'{model_key}_frac{int(round(frac * 100))}_seed{seed}'

for r in planned_runs():
    done = os.path.exists(f'{RESULTS_DIR}/{run_name(*r)}.json')
    print(run_name(*r), '✅ sudah ada' if done else '⏳ belum')

mbert_frac25_seed42 ✅ sudah ada
mbert_frac50_seed42 ✅ sudah ada
mdeberta_frac25_seed42 ✅ sudah ada
mdeberta_frac50_seed42 ✅ sudah ada


## 3. Memuat data dari notebook 01

In [6]:
import json
import pandas as pd

def read_qrels(path):
    """Baca qrels format TREC (qid Q0 docid rel) -> {qid: {docid: rel}}."""
    qrels = {}
    with open(path) as f:
        for line in f:
            p = line.split()
            if len(p) < 4:
                continue
            qrels.setdefault(str(p[0]), {})[p[2]] = int(p[3])
    return qrels

def read_run(path):
    """Baca run format TREC -> {qid: [(docid, score), ...]} terurut berdasarkan rank."""
    tmp = {}
    with open(path) as f:
        for line in f:
            qid, _, docid, rank, score, _tag = line.split()
            tmp.setdefault(qid, []).append((int(rank), docid, float(score)))
    return {q: [(d, s) for _, d, s in sorted(v)] for q, v in tmp.items()}

topics = pd.read_parquet(f'{DATA_DIR}/topics.parquet')            # qid, split, query
QUERY  = dict(zip(topics.qid, topics['query']))
with open(f'{DATA_DIR}/splits.json') as f:
    SPLITS = json.load(f)
qrels_train = read_qrels(f'{DATA_DIR}/qrels.train.tsv')
qrels_dev   = read_qrels(f'{DATA_DIR}/qrels.dev.tsv')
bm25_train  = read_run(f'{DATA_DIR}/run.bm25.train.top100.txt')
bm25_dev    = read_run(f'{DATA_DIR}/run.bm25.dev.top100.txt')
print({k: len(v) for k, v in SPLITS.items()})
print('Dev queries di run BM25:', len(bm25_dev))

{'train': 3664, 'val': 407, 'dev': 960, 'train_25': 916, 'train_50': 1832}
Dev queries di run BM25: 960


In [7]:
pairs_all = pd.read_parquet(f'{DATA_DIR}/train_pairs.parquet')
pass_df = pd.read_parquet(f'{DATA_DIR}/passages.parquet')

def passage_text(title, text):
    """Format passage untuk cross-encoder: judul artikel + isi paragraf."""
    title, text = (title or '').strip(), (text or '').strip()
    return f'{title} {text}'.strip()

PASSAGE = {d: passage_text(t, x) for d, t, x in zip(pass_df.docid, pass_df.title, pass_df.text)}

# Kandidat validasi (top-100 BM25 kueri validasi) & Dev (top-100 BM25)
val_qids = SPLITS['val']
val_cands = {q: [d for d, _ in bm25_train[q]] for q in val_qids}
dev_cands = {q: [d for d, _ in bm25_dev[q]] for q in SPLITS['dev']}
qrels_val = {q: qrels_train[q] for q in val_qids if q in qrels_train}

missing = {d for c in list(val_cands.values()) + list(dev_cands.values()) for d in c} - PASSAGE.keys()
assert not missing, f'{len(missing)} passage hilang — jalankan ulang notebook 01'
print(f'Pairs: {len(pairs_all):,} | passage: {len(PASSAGE):,} | '
      f'val: {len(val_qids)} kueri / {sum(map(len, val_cands.values())):,} pasangan | '
      f'dev: {len(dev_cands)} kueri / {sum(map(len, dev_cands.values())):,} pasangan')

Pairs: 44,814 | passage: 125,805 | val: 407 kueri / 40,358 pasangan | dev: 960 kueri / 95,310 pasangan


In [8]:
import numpy as np
import pandas as pd
import pytrec_eval

def ranking_to_run(ranking):
    """{qid: [docid urut]} -> {qid: {docid: skor}} dengan skor menurun sesuai rank,
    sehingga pytrec_eval memakai urutan persis seperti daftar (tanpa efek tie)."""
    return {q: {d: float(len(docs) - i) for i, d in enumerate(docs)}
            for q, docs in ranking.items()}

def evaluate(ranking, qrels, recall_ks=(20, 50, 100)):
    """Hitung nDCG@10, MRR@10, MAP@100, Recall@k.

    ranking : {qid: [docid, ...]} terurut (maks. 100 dokumen)
    qrels   : {qid: {docid: rel}}
    Hanya kueri yang punya >=1 dokumen relevan yang dievaluasi; kueri yang
    tidak ada di ranking diberi skor 0 (bukan dibuang).
    Return: (rata-rata dict, DataFrame per kueri)
    """
    qids = sorted(q for q, rels in qrels.items() if any(r > 0 for r in rels.values()))
    q_sub = {q: qrels[q] for q in qids}
    full = ranking_to_run({q: ranking.get(q, []) for q in qids})
    top10 = ranking_to_run({q: ranking.get(q, [])[:10] for q in qids})
    # pytrec_eval melewati kueri dengan run kosong -> isi dummy agar tetap terhitung 0
    for run in (full, top10):
        for q in qids:
            if not run[q]:
                run[q] = {'__none__': 0.0}

    rec_spec = 'recall.' + ','.join(str(k) for k in recall_ks)
    ev_full = pytrec_eval.RelevanceEvaluator(q_sub, {'ndcg_cut.10', 'map_cut.100', rec_spec})
    ev_top10 = pytrec_eval.RelevanceEvaluator(q_sub, {'recip_rank'})
    r_full, r_top10 = ev_full.evaluate(full), ev_top10.evaluate(top10)

    rows = []
    for q in qids:
        row = {'qid': q,
               'ndcg@10': r_full[q]['ndcg_cut_10'],
               'mrr@10': r_top10[q]['recip_rank'],
               'map@100': r_full[q]['map_cut_100']}
        for k in recall_ks:
            row[f'recall@{k}'] = r_full[q][f'recall_{k}']
        rows.append(row)
    per_q = pd.DataFrame(rows).set_index('qid')
    return per_q.mean().to_dict(), per_q

## 4. Komponen model: dataset, scoring, training loop

- **Input:** `tokenizer(query, passage, truncation='longest_first', max_length=256)` → `[CLS] q [SEP] p [SEP]` (untuk mDeBERTa token khususnya setara). Padding dinamis per batch.
- **Head:** `AutoModelForSequenceClassification(num_labels=1)` → satu logit dari representasi [CLS]; sigmoid(logit) = probabilitas relevan. Loss: `BCEWithLogitsLoss`.
- **Optimizer:** AdamW (weight decay tidak dikenakan pada bias & LayerNorm), jadwal linear dengan warmup 10%.
- **Seed** mengatur inisialisasi head, urutan batch, dan dropout. Pasangan latih identik untuk semua run.

In [9]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        transformers.set_seed(seed)
    except AttributeError:
        pass

class PairDataset(Dataset):
    def __init__(self, df):
        self.q = [QUERY[q] for q in df.qid]
        self.p = [PASSAGE[d] for d in df.docid]
        self.y = df.label.astype(np.float32).tolist()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.q[i], self.p[i], self.y[i]

def make_collate(tok, max_len):
    def collate(batch):
        qs, ps, ys = zip(*batch)
        enc = tok(list(qs), list(ps), truncation='longest_first', max_length=max_len,
                  padding=True, return_tensors='pt')
        enc['labels'] = torch.tensor(ys, dtype=torch.float32)
        return enc
    return collate

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=USE_BF16)

@torch.no_grad()
def score_pairs(model, tok, queries, passages, bs=HP['eval_batch_size'], max_len=HP['max_len'],
                desc='scoring'):
    """Skor (logit) untuk daftar pasangan; diurutkan berdasarkan panjang agar padding minimal."""
    model.eval()
    order = np.argsort([len(p) for p in passages])
    scores = np.empty(len(passages), dtype=np.float32)
    for i in tqdm(range(0, len(order), bs), desc=desc, leave=False):
        idx = order[i:i + bs]
        enc = tok([queries[j] for j in idx], [passages[j] for j in idx],
                  truncation='longest_first', max_length=max_len,
                  padding=True, return_tensors='pt').to(DEVICE)
        with autocast():
            logits = model(**enc).logits.squeeze(-1)
        scores[idx] = logits.float().cpu().numpy()
    return scores

def rerank(model, tok, cands, desc='rerank'):
    """cands: {qid: [docid]} -> (ranking {qid: [docid]}, DataFrame skor)."""
    rows = [(q, d, r) for q, docs in cands.items() for r, d in enumerate(docs, 1)]
    s = score_pairs(model, tok, [QUERY[q] for q, _, _ in rows], [PASSAGE[d] for _, d, _ in rows], desc=desc)
    df = pd.DataFrame(rows, columns=['qid', 'docid', 'bm25_rank'])
    df['ce_score'] = s
    ranking = {q: g.sort_values(['ce_score', 'bm25_rank'], ascending=[False, True]).docid.tolist()
               for q, g in df.groupby('qid', sort=False)}
    return ranking, df

In [10]:
def train_one(model_key, seed, frac):
    name = run_name(model_key, seed, frac)
    ckpt = MODELS[model_key]
    set_seed(seed)

    df = pairs_all if frac >= 1.0 else pairs_all[pairs_all[f'in_{int(round(frac * 100))}']]
    tok = AutoTokenizer.from_pretrained(ckpt)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=1).float().to(DEVICE)
    print('   dtype bobot:', next(model.parameters()).dtype)   # harus torch.float32
    n_params = sum(p.numel() for p in model.parameters())

    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(PairDataset(df), batch_size=HP['batch_size'], shuffle=True,
                        generator=g, collate_fn=make_collate(tok, HP['max_len']),
                        num_workers=2, drop_last=False)

    no_decay = ('bias', 'LayerNorm.weight', 'layernorm.weight', 'LayerNorm.bias')
    groups = [
        {'params': [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)],
         'weight_decay': HP['weight_decay']},
        {'params': [p for n, p in model.named_parameters() if any(k in n for k in no_decay)],
         'weight_decay': 0.0},
    ]
    opt = torch.optim.AdamW(groups, lr=HP['lr'])
    total = HP['epochs'] * len(loader)
    warm = int(HP['warmup_ratio'] * total)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / max(1, warm) if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    loss_fn = torch.nn.BCEWithLogitsLoss()

    print(f'\n=== {name} | {ckpt} | {n_params/1e6:.0f}M param | {len(df):,} pasangan '
          f'({df.qid.nunique():,} kueri) | {total:,} step ===')
    history, best = [], {'ndcg@10': -1}
    step, t0 = 0, time.time()
    for epoch in range(1, HP['epochs'] + 1):
        model.train()
        run_loss, n_loss = 0.0, 0
        for batch in tqdm(loader, desc=f'epoch {epoch}', leave=False):
            labels = batch.pop('labels').to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with autocast():
                logits = model(**batch).logits.squeeze(-1)
            loss = loss_fn(logits.float(), labels)
            if not torch.isfinite(loss):
                raise RuntimeError(f'Loss NaN di step {step + 1} -> training dihentikan')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), HP['max_grad_norm'])
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            step += 1
            run_loss += loss.item(); n_loss += 1
            if step % LOG_EVERY == 0:
                history.append({'step': step, 'train_loss': run_loss / n_loss,
                                'lr': sched.get_last_lr()[0]})
                run_loss, n_loss = 0.0, 0

        # --- validasi akhir epoch: rerank top-100 BM25 kueri validasi ---
        val_rank, val_df = rerank(model, tok, val_cands, desc='validasi')
        assert val_df.ce_score.notna().all() and val_df.ce_score.nunique() > 1, \
            'Skor validasi NaN/konstan -> hentikan'
        val_metrics, _ = evaluate(val_rank, qrels_val)
        history.append({'step': step, 'epoch': epoch, **{f'val_{k}': v for k, v in val_metrics.items()}})
        print(f'epoch {epoch}: val nDCG@10 = {val_metrics["ndcg@10"]:.4f} | '
              f'MRR@10 = {val_metrics["mrr@10"]:.4f} | {(time.time() - t0)/60:.1f} menit')
        if val_metrics['ndcg@10'] > best['ndcg@10']:
            best = {'epoch': epoch, **val_metrics}
            best_state = {k: v.detach().to('cpu', copy=True) for k, v in model.state_dict().items()}
    train_minutes = (time.time() - t0) / 60

    # --- muat checkpoint terbaik, rerank Dev ---
    model.load_state_dict(best_state)
    del best_state
    dev_rank, dev_scores = rerank(model, tok, dev_cands, desc='Dev')
    dev_scores['bm25_score'] = [s for q in dev_cands for _, s in bm25_dev[q]]
    dev_metrics, dev_perq = evaluate(dev_rank, qrels_dev)
    print(f'>>> Dev (k=100): ' + ' | '.join(f'{k} {v:.4f}' for k, v in dev_metrics.items()))

    dev_scores.to_parquet(f'{SCORES_DIR}/{name}.dev_top100.parquet', index=False)
    dev_perq.to_csv(f'{RESULTS_DIR}/{name}.per_query.csv')
    result = {
        'run': name, 'model_key': model_key, 'checkpoint': ckpt, 'seed': seed, 'train_frac': frac,
        'n_params': n_params, 'n_pairs': int(len(df)), 'n_train_queries': int(df.qid.nunique()),
        'hp': HP, 'best_epoch': best['epoch'], 'val_best': best, 'dev_k100': dev_metrics,
        'train_minutes': train_minutes, 'gpu': GPU_NAME, 'bf16': USE_BF16,
        'torch': torch.__version__, 'transformers': transformers.__version__,
        'history': history,
    }
    with open(f'{RESULTS_DIR}/{name}.json', 'w') as f:
        json.dump(result, f, indent=2)
    return model, tok, result

## 5. Pengukuran efisiensi (Eksp. 2, seed 42)

Protokol (slide 12): latency per kueri **termasuk tokenisasi**, batch size inferensi tetap (32), 20 kueri pertama sebagai warmup dibuang, `torch.cuda.synchronize()` sebelum/ sesudah pengukuran. Diukur pada seluruh 960 kueri Dev untuk setiap k. Memori = puncak memori GPU yang dialokasikan selama inferensi.

In [11]:
@torch.no_grad()
def benchmark(model, tok, k_list=EFF_K_LIST, bs=EFF_INFER_BS, n_warmup=EFF_WARMUP_QUERIES):
    model.eval()
    qids = list(dev_cands)
    out = {}
    for k in k_list:
        texts = {q: (QUERY[q], [PASSAGE[d] for d in dev_cands[q][:k]]) for q in qids}
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        times = []
        for i, q in enumerate(tqdm(qids, desc=f'latency k={k}', leave=False)):
            query, ps = texts[q]
            torch.cuda.synchronize(); t0 = time.perf_counter()
            for j in range(0, len(ps), bs):
                chunk = ps[j:j + bs]
                enc = tok([query] * len(chunk), chunk, truncation='longest_first',
                          max_length=HP['max_len'], padding=True, return_tensors='pt').to(DEVICE)
                with autocast():
                    model(**enc).logits
            torch.cuda.synchronize()
            if i >= n_warmup:
                times.append(time.perf_counter() - t0)
        t = np.array(times) * 1000
        n_pairs = sum(len(texts[q][1]) for q in qids[n_warmup:])
        out[k] = {
            'latency_ms_mean': float(t.mean()), 'latency_ms_std': float(t.std()),
            'latency_ms_p50': float(np.percentile(t, 50)), 'latency_ms_p95': float(np.percentile(t, 95)),
            'throughput_pairs_per_s': float(n_pairs / (t.sum() / 1000)),
            'throughput_queries_per_s': float(len(t) / (t.sum() / 1000)),
            'peak_gpu_mem_mb': torch.cuda.max_memory_allocated() / 2**20,
            'n_queries_timed': int(len(t)),
        }
        print(f'k={k:>3}: {out[k]["latency_ms_mean"]:.1f} ms/kueri (p95 {out[k]["latency_ms_p95"]:.1f}) | '
              f'{out[k]["throughput_pairs_per_s"]:.0f} pasangan/s | '
              f'puncak memori {out[k]["peak_gpu_mem_mb"]:.0f} MB')
    return out

## 6. Jalankan eksperimen

In [12]:
for model_key, seed, frac in planned_runs():
    name = run_name(model_key, seed, frac)
    eff_path = f'{RESULTS_DIR}/efficiency_{model_key}.json'
    need_eff = (seed == 42 and frac >= 1.0 and DEVICE.type == 'cuda'
                and (FORCE_RERUN or not os.path.exists(eff_path)))
    if os.path.exists(f'{RESULTS_DIR}/{name}.json') and not FORCE_RERUN and not need_eff:
        print(f'⏭️  {name} sudah ada, dilewati'); continue
    if os.path.exists(f'{RESULTS_DIR}/{name}.json') and not FORCE_RERUN and need_eff:
        print(f'⚠️  {name} sudah ada tetapi efisiensi belum diukur — training diulang agar bobot seed 42 tersedia')

    model, tok, res = train_one(model_key, seed, frac)
    if need_eff:
        eff = benchmark(model, tok)
        with open(eff_path, 'w') as f:
            json.dump({'model_key': model_key, 'checkpoint': MODELS[model_key], 'run': name,
                       'gpu': GPU_NAME, 'bf16': USE_BF16, 'infer_batch_size': EFF_INFER_BS,
                       'max_len': HP['max_len'], 'n_params': res['n_params'],
                       'weights_mem_mb': sum(p.numel() * p.element_size() for p in model.parameters()) / 2**20,
                       'by_k': eff}, f, indent=2)
    del model, tok
    gc.collect(); torch.cuda.empty_cache()

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

epoch 3:   0%|          | 0/1404 [00:00<?, ?it/s]

validasi:   0%|          | 0/316 [00:00<?, ?it/s]

epoch 3: val nDCG@10 = 0.5830 | MRR@10 = 0.6821 | 30.7 menit


Dev:   0%|          | 0/745 [00:00<?, ?it/s]

>>> Dev (k=100): ndcg@10 0.5891 | mrr@10 0.6895 | map@100 0.5137 | recall@20 0.7870 | recall@50 0.8786 | recall@100 0.9041


## 7. Ringkasan run yang sudah selesai

In [13]:
rows = []
for fn in sorted(os.listdir(RESULTS_DIR)):
    if fn.endswith('.json') and '_frac' in fn:
        r = json.load(open(f'{RESULTS_DIR}/{fn}'))
        rows.append({'run': r['run'], 'model': r['model_key'], 'frac': r['train_frac'], 'seed': r['seed'],
                     'best_epoch': r['best_epoch'], 'val_ndcg@10': r['val_best']['ndcg@10'],
                     **{f'dev_{k}': v for k, v in r['dev_k100'].items()},
                     'train_min': r['train_minutes']})
summary = pd.DataFrame(rows).sort_values(['frac', 'model', 'seed'], ascending=[False, True, True])
bm25 = json.load(open(f'{RESULTS_DIR}/bm25_dev_metrics.json'))
print('BM25 Dev nDCG@10 =', round(bm25['ndcg@10'], 4))
summary.round(4)

BM25 Dev nDCG@10 = 0.4486


,run,model,frac,seed,best_epoch,val_ndcg@10,dev_ndcg@10,dev_mrr@10,dev_map@100,dev_recall@20,dev_recall@50,dev_recall@100,train_min
2,mbert_frac100_seed42,mbert,1.00,42,2,0.6085,0.5925,0.6917,0.5143,0.7901,0.8825,0.9041,25.2541
0,mbert_frac100_seed100,mbert,1.00,100,3,0.5945,0.5906,0.6928,0.5117,0.7824,0.8788,0.9041,25.0540
1,mbert_frac100_seed2026,mbert,1.00,2026,2,0.6017,0.6011,0.7024,0.5215,0.7874,0.8782,0.9041,25.0542
7,mdeberta_frac100_seed42,mdeberta,1.00,42,2,0.6058,0.5983,0.6980,0.5238,0.7944,0.8848,0.9041,53.4044
5,mdeberta_frac100_seed100,mdeberta,1.00,100,3,0.6062,0.6100,0.7101,0.5360,0.7990,0.8800,0.9041,53.3854
6,mdeberta_frac100_seed2026,mdeberta,1.00,2026,3,0.6116,0.6022,0.7079,0.5305,0.7845,0.8836,0.9041,53.4030
4,mbert_frac50_seed42,mbert,0.50,42,2,0.5818,0.5725,0.6700,0.4960,0.7807,0.8744,0.9041,14.6162
9,mdeberta_frac50_seed42,mdeberta,0.50,42,3,0.5830,0.5891,0.6895,0.5137,0.7870,0.8786,0.9041,30.7238
3,mbert_frac25_seed42,mbert,0.25,42,3,0.5486,0.5492,0.6468,0.4703,0.7501,0.8650,0.9041,9.3016
8,mdeberta_frac25_seed42,mdeberta,0.25,42,2,0.5838,0.5715,0.6694,0.4950,0.7767,0.8713,0.9041,19.3419


In [14]:
# Rata-rata ± simpangan baku 3 seed (Eksp. 1)
exp1 = summary[summary.frac == 1.0]
if len(exp1):
    cols = [c for c in exp1.columns if c.startswith('dev_')]
    agg = exp1.groupby('model')[cols].agg(['mean', 'std'])
    display(agg.round(4))

dev_ndcg@10         dev_mrr@10         dev_map@100          \
                mean     std       mean     std        mean     std   
model                                                                 
mbert         0.5947  0.0056     0.6956  0.0058      0.5159  0.0051   
mdeberta      0.6035  0.0060     0.7053  0.0064      0.5301  0.0061   

         dev_recall@20         dev_recall@50         dev_recall@100       
                  mean     std          mean     std           mean  std  
model                                                                     
mbert           0.7867  0.0039        0.8799  0.0023         0.9041  0.0  
mdeberta        0.7926  0.0074        0.8828  0.0025         0.9041  0.0